In [12]:
import pandas as pd
import numpy as np
import pyreadr
from word2number import w2n

#Before Cleaning

In [46]:
raw_df = pyreadr.read_r("/content/drive/MyDrive/Ch2_Exercise1_DonutX.RData")["dta"]

In [47]:
df = raw_df.copy()
df

,name,donutsx,weightx,childx,malex
0,Homer,14,275.0,0.0,0.0
1,Marge,0,141.0,0.0,1.0
2,Lisa,0,70.0,-1.0,1.0
3,Bart,5,75.0,-1.0,0.0
4,ComicBookGuy,20,310.0,0.0,0.0
5,Mr Burns,0.75,80.0,0.0,0.0
6,Smithers,0.25,160.0,0.0,0.0
7,Chief Wiggum,16,263.0,0.0,0.0
8,Principal Skinner,3,205.0,0.0,0.0
9,Rev. Lovejoy,2,185.0,0.0,0.0


In [48]:
df.dtypes

,0
name,object
donutsx,object
weightx,float64
childx,float64
malex,float64


In [49]:
df.describe()

,weightx,childx,malex
count,13.000000,13.000000,13.000000
mean,279.153846,-0.153846,0.307692
std,389.329091,0.375534,0.480384
min,70.000000,-1.000000,0.000000
25%,141.000000,0.000000,0.000000
50%,170.000000,0.000000,0.000000
75%,263.000000,0.000000,1.000000
max,1550.000000,0.000000,1.000000


#Cleaning

In [50]:
#Convert donuts column to numeric and use word2number library to convert rows that failed.
numeric_donuts = pd.to_numeric(df["donutsx"], errors="coerce")

failed_mask = numeric_donuts.isna() & df["donutsx"].notna()

def parse_number(value):
    try:
        return float(w2n.word_to_num(str(value).strip().lower()))
    except (ValueError, TypeError):
        return pd.NA

numeric_donuts.loc[failed_mask] = (
    df.loc[failed_mask, "donutsx"]
      .apply(parse_number)
)

df["donutsx"] = numeric_donuts

In [52]:
#Convert boolean/dummy variables into non-negative values

booleans = ['childx', 'malex']

df[booleans] = np.abs(df[booleans])

In [56]:
# Update Patty's "fat-fingered" weight to 155 -
# In practice, I would prefer to treat this as an outlier and remove it from the dataset completely
df['weightx'] = np.where(df['name']=='Patty', 155, df['weightx'])

In [62]:
# Rename Male Column to Female
df = df.rename(columns={"malex": "femalex"})

#After Cleaning

In [63]:
df

,name,donutsx,weightx,childx,femalex
0,Homer,14.00,275.0,0.0,0.0
1,Marge,0.00,141.0,0.0,1.0
2,Lisa,0.00,70.0,1.0,1.0
3,Bart,5.00,75.0,1.0,0.0
4,ComicBookGuy,20.00,310.0,0.0,0.0
5,Mr Burns,0.75,80.0,0.0,0.0
6,Smithers,0.25,160.0,0.0,0.0
7,Chief Wiggum,16.00,263.0,0.0,0.0
8,Principal Skinner,3.00,205.0,0.0,0.0
9,Rev. Lovejoy,2.00,185.0,0.0,0.0


In [58]:
df.describe()

,donutsx,weightx,childx,femalex
count,13.000000,13.000000,13.000000,13.000000
mean,5.446154,171.846154,0.153846,0.307692
std,6.749551,76.155374,0.375534,0.480384
min,0.000000,70.000000,0.000000,0.000000
25%,0.750000,141.000000,0.000000,0.000000
50%,3.000000,160.000000,0.000000,0.000000
75%,5.000000,205.000000,0.000000,1.000000
max,20.000000,310.000000,1.000000,1.000000
